In [2]:
import duckdb
import pandas as pd
import polars as pl
import time

print("All libraries imported successfully!")

All libraries imported successfully!


In [3]:
con = duckdb.connect()

df = con.execute("""
SELECT *
FROM read_csv_auto('indian_business_dataset.csv')
""").df()

print(df.head())

  business_id     business_name    owner_name           state       city  age  \
0        B001       Sri Traders    Ravi Kumar  Andhra Pradesh   Tirupati   35   
1        B002        ABC Stores     Anita Rao       Telangana  Hyderabad   42   
2        B003  Lakshmi Textiles  Lakshmi Devi       Karnataka  Bengaluru   29   
3        B004        Green Mart  Suresh Reddy      Tamil Nadu    Chennai   51   
4        B005  Ravi Enterprises  Priya Sharma     Maharashtra       Pune   38   

                email       phone  annual_revenue            gstin  
0  owner1@example.com  9000000001          500000  37ABCDE0001F1Z5  
1  owner2@example.com  9000000002          750000  37ABCDE0002F1Z5  
2  owner3@example.com  9000000003          420000  37ABCDE0003F1Z5  
3  owner4@example.com  9000000004          900000  37ABCDE0004F1Z5  
4                 NaN  9000000005          650000  37ABCDE0005F1Z5  


In [4]:
print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns)

print("\nDataset Information:")
df.info()

Dataset Shape: (50, 10)

Column Names:
Index(['business_id', 'business_name', 'owner_name', 'state', 'city', 'age',
       'email', 'phone', 'annual_revenue', 'gstin'],
      dtype='str')

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   business_id     50 non-null     str  
 1   business_name   50 non-null     str  
 2   owner_name      50 non-null     str  
 3   state           50 non-null     str  
 4   city            50 non-null     str  
 5   age             50 non-null     int64
 6   email           49 non-null     str  
 7   phone           50 non-null     int64
 8   annual_revenue  50 non-null     int64
 9   gstin           50 non-null     str  
dtypes: int64(3), str(7)
memory usage: 4.0 KB


In [5]:
result = con.execute("""
SELECT COUNT(*) AS total_businesses
FROM read_csv_auto('indian_business_dataset.csv')
""").df()

print(result)

   total_businesses
0                50


In [6]:
result = con.execute("""
SELECT SUM(annual_revenue) AS total_revenue
FROM read_csv_auto('indian_business_dataset.csv')
""").df()

print(result)

   total_revenue
0     31450000.0


In [7]:
result = con.execute("""
SELECT AVG(annual_revenue) AS average_revenue
FROM read_csv_auto('indian_business_dataset.csv')
""").df()

print(result)

   average_revenue
0         629000.0


In [8]:
result = con.execute("""
SELECT state, SUM(annual_revenue) AS total_revenue
FROM read_csv_auto('indian_business_dataset.csv')
GROUP BY state
ORDER BY total_revenue DESC
""").df()

print(result)

            state  total_revenue
0      Tamil Nadu      8050000.0
1  Andhra Pradesh      6600000.0
2       Telangana      6550000.0
3     Maharashtra      6250000.0
4       Karnataka      4000000.0


In [9]:
result = con.execute("""
SELECT business_name, state, annual_revenue
FROM read_csv_auto('indian_business_dataset.csv')
ORDER BY annual_revenue DESC
LIMIT 5
""").df()

print(result)

  business_name       state  annual_revenue
0    Green Mart  Tamil Nadu          900000
1    Green Mart  Tamil Nadu          900000
2    Green Mart  Tamil Nadu          900000
3    Green Mart  Tamil Nadu          900000
4    Green Mart  Tamil Nadu          900000


In [10]:
start = time.time()

duck_result = con.execute("""
SELECT state, SUM(annual_revenue) AS total_revenue
FROM read_csv_auto('indian_business_dataset.csv')
GROUP BY state
ORDER BY total_revenue DESC
""").df()

duck_time = time.time() - start

print(duck_result)
print("DuckDB Time:", duck_time, "seconds")

            state  total_revenue
0      Tamil Nadu      8050000.0
1  Andhra Pradesh      6600000.0
2       Telangana      6550000.0
3     Maharashtra      6250000.0
4       Karnataka      4000000.0
DuckDB Time: 0.004987955093383789 seconds


In [12]:
start = time.time()

pandas_result = (
    df.groupby('state')['annual_revenue']
    .sum()
    .sort_values(ascending=False)
)

pandas_time = time.time() - start

print(pandas_result)
print("Pandas Time:", pandas_time)

state
Tamil Nadu        8050000
Andhra Pradesh    6600000
Telangana         6550000
Maharashtra       6250000
Karnataka         4000000
Name: annual_revenue, dtype: int64
Pandas Time: 0.0


In [13]:
start = time.time()

duck_result = con.execute("""
SELECT state, SUM(annual_revenue) AS total_revenue
FROM read_csv_auto('indian_business_dataset.csv')
GROUP BY state
ORDER BY total_revenue DESC
""").df()

duck_time = time.time() - start

print("DuckDB Time:", duck_time)

DuckDB Time: 0.005415439605712891


In [16]:
start = time.time()

polars_result = (
    pl_df.group_by("state")
    .agg(pl.col("annual_revenue").sum().alias("total_revenue"))
    .sort("total_revenue", descending=True)
)

polars_time = time.time() - start

print("Polars Time:", polars_time)

Polars Time: 0.002269268035888672


In [15]:
import polars as pl

pl_df = pl.read_csv("indian_business_dataset.csv")

print(pl_df.head())

shape: (5, 10)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ business_ ┆ business_ ┆ owner_nam ┆ state     ┆ … ┆ email     ┆ phone     ┆ annual_re ┆ gstin    │
│ id        ┆ name      ┆ e         ┆ ---       ┆   ┆ ---       ┆ ---       ┆ venue     ┆ ---      │
│ ---       ┆ ---       ┆ ---       ┆ str       ┆   ┆ str       ┆ i64       ┆ ---       ┆ str      │
│ str       ┆ str       ┆ str       ┆           ┆   ┆           ┆           ┆ i64       ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ B001      ┆ Sri       ┆ Ravi      ┆ Andhra    ┆ … ┆ owner1@ex ┆ 900000000 ┆ 500000    ┆ 37ABCDE0 │
│           ┆ Traders   ┆ Kumar     ┆ Pradesh   ┆   ┆ ample.com ┆ 1         ┆           ┆ 001F1Z5  │
│ B002      ┆ ABC       ┆ Anita Rao ┆ Telangana ┆ … ┆ owner2@ex ┆ 900000000 ┆ 750000    ┆ 37ABCDE0 │
│           ┆ Stores    ┆           ┆           ┆   ┆ ample.com ┆ 2         

In [17]:
start = time.time()

polars_result = (
    pl_df.group_by("state")
    .agg(
        pl.col("annual_revenue").sum().alias("total_revenue")
    )
    .sort("total_revenue", descending=True)
)

polars_time = time.time() - start

print(polars_result)
print("Polars Time:", polars_time, "seconds")

shape: (5, 2)
┌────────────────┬───────────────┐
│ state          ┆ total_revenue │
│ ---            ┆ ---           │
│ str            ┆ i64           │
╞════════════════╪═══════════════╡
│ Tamil Nadu     ┆ 8050000       │
│ Andhra Pradesh ┆ 6600000       │
│ Telangana      ┆ 6550000       │
│ Maharashtra    ┆ 6250000       │
│ Karnataka      ┆ 4000000       │
└────────────────┴───────────────┘
Polars Time: 0.00101470947265625 seconds


In [18]:
comparison = pd.DataFrame({
    "Tool": ["Pandas", "DuckDB", "Polars"],
    "Execution Time": [
        pandas_time,
        duck_time,
        polars_time
    ]
})

print(comparison)

     Tool  Execution Time
0  Pandas        0.000000
1  DuckDB        0.005415
2  Polars        0.001015
